In [115]:
import pandas as pd

In [181]:
# Exposure scores
df = pd.read_csv('../data/occ_level.csv')

# O*NET 8 digit to SOC 6 digit crosswalk
crosswalk = pd.read_excel('../data/2019_to_SOC_crosswalk.xlsx', skiprows=3)

# BLS SOC 6 digit to Census/CPS crosswalk
census = pd.read_excel('../data/nem-occcode-cps-crosswalk.xlsx', sheet_name="NEM SOC CPS crosswalk", skiprows=4)

# BLS employment data for weighting
bls = pd.read_excel('../data/national_M2024_dl.xlsx', sheet_name='national_M2024_dl', dtype=str)

In [170]:
print(df['O*NET-SOC Code'].nunique())
# 923 unique occupation codes at 8-digit granularity

923


In [171]:
# Crosswalk O*NET to SOC
merged = df.merge(crosswalk, left_on='O*NET-SOC Code', right_on='O*NET-SOC 2019 Code', how='left', indicator=True)

print(merged['O*NET-SOC 2019 Code'].nunique())
print(merged['2018 SOC Code'].nunique())
# after merge 125 8-digit codes are collapsed into 6-digit codes

923
798


In [ ]:
# Flag duplicates and calculate group-level standard deviations for those collapsed at this stage
merged['many_to_one_O*NET'] = merged.duplicated(subset=['2018 SOC Code'], keep=False)
cols = ['dv_rating_alpha', 'dv_rating_beta', 'dv_rating_gamma']

std_df = merged.groupby('2018 SOC Code')[cols].std().reset_index()
std_df = std_df.rename(columns={col: f'{col}_std' for col in cols})

merged_with_stats = merged.merge(std_df, on='2018 SOC Code', how='left')

# Export pre-merge rows that were part of a many-to-one collapse
multi_code_rows = merged_with_stats[merged_with_stats['many_to_one_O*NET'] == True]
multi_code_rows.to_csv('../temp/pre_merge_collapsed_O*NET.csv', index=False)

# Build the aggregation dictionary for the untouched final grouped df
agg_dict = {col: 'mean' for col in cols}
agg_dict['many_to_one_O*NET'] = 'first'
agg_dict['2018 SOC Title'] = 'first'

grouped = merged.groupby('2018 SOC Code', as_index=False).agg(agg_dict)

In [ ]:
# Alpha, Beta, and Gamma scores have intra-group std of 0.088, 0.076, 0.099 after merge.
# 0.42 is the highest sigma for alpha scores (Computer Systems Analysts + Health Informatics Specialists)
# 15-1211.00 + 15-1211.01
# 0.34 is the highest sigma for beta scores (Speech-Language Pathology Assistants + Endoscopy Technicians)
# 31-9099.01 + 31-9099.02
# 0.51 is the highest sigma for gamma scores (Speech-Language Pathology Assistants + Endoscopy Technicians)

In [179]:
print(grouped.columns)
print(grouped.shape[0])
print(grouped['2018 SOC Code'].nunique())
# 798 unique 6 digit codes remain

Index(['2018 SOC Code', 'dv_rating_alpha', 'dv_rating_beta', 'dv_rating_gamma',
       'many_to_one_O*NET', '2018 SOC Title'],
      dtype='str')
798
798


In [122]:
census = pd.read_excel('../data/nem-occcode-cps-crosswalk.xlsx', sheet_name="NEM SOC CPS crosswalk", skiprows=4)

In [70]:
print(census.columns)

Index(['Sort Order', 'National Employment Matrix code',
       'National Employment Matrix title', 'CPS code', 'CPS occupation title'],
      dtype='str')


In [154]:
merged2 = grouped.merge(census, left_on="2018 SOC Code", right_on = "National Employment Matrix code", how="left", indicator=True)

In [ ]:
print(merged2['_merge'].value_counts())
print(merged2['CPS code'].isna().sum())
# 25 SOC codes do not have a Census equivalent in the BLS crosswalk

print(merged2['2018 SOC Code'].nunique())
print(merged2['CPS code'].nunique())
# 503 unique census codes after the merge

_merge
both          773
left_only      25
right_only      0
Name: count, dtype: int64
25
798
503


In [ ]:
# Prepare lowercase title columns for matching
merged2['title_str'] = merged2['2018 SOC Title'].str.strip().str.lower()
census['title_str'] = census['CPS occupation title'].str.strip().str.lower()

# Rename the lookup column to avoid naming conflicts
title_lookup = census.rename(columns={'CPS code': 'CPS from title'})

# Merge the lookup data back into merged2 based on the title string
merged_temp = merged2.merge(title_lookup[['title_str', 'CPS from title']], on='title_str', how='left', indicator='_merge2')

# Fill missing 'CPS code' values in merged2 using the matched lookup values
merged2['CPS code'] = merged2['CPS code'].fillna(merged_temp['CPS from title'])

# Check how many missing values remain in 'CPS code'
print(merged2['CPS code'].isna().sum())

18


In [158]:
success = merged_temp[(merged_temp['_merge2'] == 'both') & (merged_temp['CPS code'].isna())]
success.to_csv('../temp/merge.csv', index=False)

failed = merged2[merged2['CPS code'].isna()]
failed.to_csv('../temp/fail.csv', index=False)

In [153]:
print(merged2['CPS code'].isna().sum())

18


In [ ]:
merged2.to_csv('../temp.output.csv')    

In [230]:
direct_mappings = {
    '13-2022': '810', # Appraisers of Personal and Business Property, 13-2020
    '13-2023': '810', # Appraisers of Personal and Business Property, 13-2020
    '25-2055': '2330', # Special Education Teachers, Kindergarten, 25-2052
    '25-2056': '2330', # Special Education Teachers, Elementary School, 25-2052
    '25-9042': '2545', # Teaching Assistants, Preschool, Elementary, Middle, and Secondary School, Except Special Education 25-9045
    '25-9043': '2545', # Teaching Assistants, Special Education 25-9045
    '29-2011': '3300', # Medical and Clinical Laboratory Technologists 29-2010
    '29-2012': '3300', # Medical and Clinical Laboratory Technicians 29-2010
    '39-7011': '4540', # Tour Guides and Escorts 39-7010
    '39-7012': '4540', # Travel Guides 39-7010
    '47-4091': '6765', # Segmental Pavers 47-4090
    '47-4099': '6765', # Construction and Related Workers, All Other 47-4090
    '51-2022': '7720', # Electrical and Electronic Equipment Assemblers 51-2028
    '51-2023': '7720', # Electromechanical Equipment Assemblers 51-2028
    '51-2092': '7750', # Team Assemblers 51-2090
    '53-1042': '9005', # First-Line Supervisors of Helpers, Laborers, and Material Movers, Hand 53-1047
    '53-1043': '9005', # First-Line Supervisors of Material-Moving Machine and Vehicle Operators 53-1047 
    '53-1044': '9005' # First-Line Supervisors of Passenger Attendants 53-1047
}

merged2['CPS code'] = merged2['CPS code'].fillna(merged2['2018 SOC Code'].map(direct_mappings))


# Also map to the BLS Employment matrix because that is what we will weight via
bls_mappings = {
    '13-2022': '13-2020',
    '13-2023': '13-2020',
    '25-2055': '25-2052',
    '25-2056': '25-2052',
    '25-9042': '25-9045',
    '25-9043': '25-9045',
    '29-2011': '29-2010',
    '29-2012': '29-2010',
    '39-7011': '39-7010',
    '39-7012': '39-7010',
    '47-4091': '47-4090',
    '47-4099': '47-4090',
    '51-2022': '51-2028',
    '51-2023': '51-2028',
    '51-2092': '51-2090',
    '53-1042': '53-1047',
    '53-1043': '53-1047',
    '53-1044': '53-1047', # Below this is relabelling successful merges to work with BLS, will update with names for reference later
    '13-1021': '13-1020',
    '13-1022': '13-1020',
    '13-1023': '13-1020',
    '21-1011': '21-1018',
    '21-1014': '21-1018',
    '31-1121': '31-1120',
    '31-1122': '31-1120',
}

merged2['BLS code'] = merged2['2018 SOC Code'].map(bls_mappings).fillna(merged2['2018 SOC Code'])

In [224]:
print(merged2.columns)

Index(['2018 SOC Code', 'dv_rating_alpha', 'dv_rating_beta', 'dv_rating_gamma',
       'many_to_one', '2018 SOC Title', 'Sort Order',
       'National Employment Matrix code', 'National Employment Matrix title',
       'CPS code', 'CPS occupation title', 'title_str', '_merge', 'BLS code'],
      dtype='str')


In [231]:
bls['TOT_EMP'] = pd.to_numeric(bls['TOT_EMP'], errors='coerce')

merged3 = merged2.merge(bls[['OCC_CODE', 'TOT_EMP']], left_on='BLS code', right_on='OCC_CODE', how='left', indicator='_merge3')

print(merged3[merged3['TOT_EMP'].isna()]['BLS code'])

merged3.to_csv('../temp/merge3.csv')


547    45-3031
Name: BLS code, dtype: str


In [232]:

merged3['emp_share'] = merged3.groupby('CPS code')['TOT_EMP'].transform(lambda x: x / x.sum())

In [236]:


# 2. Flag duplicates and calculate group-level standard deviations using unweighted std
merged3['many_to_one_OCC'] = merged3.duplicated(subset=['CPS code'], keep=False)
cols = ['dv_rating_alpha', 'dv_rating_beta', 'dv_rating_gamma']

std_df = merged3.groupby('CPS code')[cols].std().reset_index()
std_df = std_df.rename(columns={col: f'{col}_std' for col in cols})

merged3_with_stats = merged3.merge(std_df, on='CPS code', how='left')

# 3. Export pre-merge rows that were part of a many-to-one collapse
multi_code_occ_rows = merged3_with_stats[merged3_with_stats['many_to_one_OCC'] == True]
multi_code_occ_rows.to_csv('../temp/pre_merge_cps_collapsed_rows.csv', index=False)

# 4. Weighted average for each rating column (leaving df_agg untouched by std dev)
name_col = '2018 SOC Code'  # <-- set this to your actual occupation-name/SOC column

for col in cols:
    merged3[col] = pd.to_numeric(merged3[col], errors='coerce')
    merged3[f'{col}_weighted'] = merged3[col] * merged3['emp_share']

agg_dict = {f'{col}_weighted': 'sum' for col in cols}

df_agg = merged3.groupby('CPS code', as_index=False).agg(agg_dict)

# collect the names that were merged into each CPS code
merged_names = (
    merged3.groupby('CPS code')[name_col]
    .agg(lambda x: '; '.join(sorted(set(x.astype(str)))))
    .reset_index()
    .rename(columns={name_col: 'merged_from'})
)

df_agg = df_agg.merge(merged_names, on='CPS code', how='left')

# 5. Rename back
df_agg = df_agg.rename(columns={f'{col}_weighted': col for col in cols})

In [237]:
print(df_agg.shape[0])
print(df_agg.isna().sum())

519
CPS code           0
dv_rating_alpha    0
dv_rating_beta     0
dv_rating_gamma    0
merged_from        0
dtype: int64


In [238]:
df_agg.to_csv('../data/final_exposure.csv', index=False)